# 03 · Expense forecasting

Compares Linear Regression and Random Forest Regression for:

- total monthly expenses
- each of the eight expense categories

Validation is chronological: the most recent 12 months are held out and never
used for fitting. Metrics reported are MAE, RMSE and R². The best estimator per
target is persisted to `backend/ml/expense_forecasting.pkl`.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "ml"))
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import json

import joblib
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
import train_forecasting as tf

df = tf.load_data()
print(f"Monthly rows: {len(df)}")
df.tail()


## History used by the forecast

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
df.plot(x="date", y="total_expenses", ax=ax, marker="o", color="#2563EB", legend=False)
ax.set_title("Monthly total expenses")
ax.set_ylabel("Amount (INR)")
ax.set_xlabel("")
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for column in tf.CATEGORY_COLUMNS:
    df.plot(x="date", y=column, ax=ax, linewidth=1.6, label=column.replace("_expenses", ""))
ax.set_title("Category spending history")
ax.set_ylabel("Amount (INR)")
ax.set_xlabel("")
ax.legend(ncol=4, fontsize=8)
fig.tight_layout()
plt.show()


## Train and validate

`tf.main()` fits both estimators per target, selects the best by R² (lower RMSE
as tie-breaker), saves the validation chart and writes the model bundle.


In [ ]:
tf.main()


## Metric comparison

In [ ]:
metrics = json.loads((PROJECT_ROOT / "backend" / "ml" / "forecasting_metrics.json").read_text(encoding="utf-8"))

rows = []
for target, candidates in metrics["comparison"].items():
    for model_name, scores in candidates.items():
        rows.append({"target": target, "model": model_name, **scores})

results = pd.DataFrame(rows)
results.sort_values(["target", "model"]).reset_index(drop=True)


In [ ]:
pd.Series(metrics["best_models"], name="best_model")


## Validation chart: last 12 months held out

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(PROJECT_ROOT / "backend" / "ml" / "forecast_validation.png")))


## Persisted bundle

The API blends this global seasonal model with the signed-in user's recent
history. If a user has fewer than three months of expenses, or no expenses in
the current month, the API returns an explicit insufficient-data message instead
of a fabricated number.


In [ ]:
bundle = joblib.load(PROJECT_ROOT / "backend" / "ml" / "expense_forecasting.pkl")
{
    "trained_at": bundle["trained_at"],
    "training_rows": bundle["training_rows"],
    "validation_rows": bundle["validation_rows"],
    "next_month": f"{bundle['next_year']}-{bundle['next_month']:02d}",
    "best_models": {target: info["model_name"] for target, info in bundle["models"].items()},
}


## Notes for the report

- The model learns seasonality through `month_sin`, `month_cos` and `quarter`,
  plus a linear time index.
- Random Forest can capture non-linear patterns but is limited by the small
  monthly dataset, so Linear Regression often wins on the total target.
- Forecasts are clearly labelled as **estimates** in the UI, reports and
  assistant replies.
- Retrain any time with `python backend/ml/train_forecasting.py`.
